# Task 11 — Thực nghiệm Tham chiếu `XLM-R-base` (`30` train + `6` `ZS` + Benchmark `ViSoBERT` vs `XLMR`, `phase2-v1`)

Chọn **Runtime → Change runtime type → GPU** (ưu tiên **`NVIDIA A100`** hoặc **`NVIDIA L4`** hỗ trợ `BF16 strict`), kiểm tra các đường dẫn ở **Bước 1**, rồi chạy lần lượt từng cell theo thứ tự.

### Phạm vi thực nghiệm & Thứ tự phụ thuộc (`phase2-v1`)
- **Mục tiêu:** Chạy trọn vẹn gói tham chiếu **`xlm-roberta-base`** (`seed 0..2`) trên cùng một commit và cùng profile môi trường **`colab-task06-v1`**:
  1. **Smoke test (`dev-only`, `50` mẫu)** vào thư mục tách biệt `/content/task11_xlmr_smoke`.
  2. **6 lượt `DFT sweep` (`10%`, `seed 0`, `dev-only`)** → khóa `selection_xlmr_dft_lr.json`.
  3. **6 lượt `DFT 100%` (`seed 0..2`, `save_ckpt=True`)** dùng `--selection-path selection_xlmr_dft_lr.json` làm mốc 100% và checkpoint nguồn cho `Seq`/`ZS`.
  4. **6 lượt `Seq sweep` (`10%`, `seed 0`, `dev-only`)** nạp checkpoint `DFT 100%` → khóa `selection_xlmr_lr.json`.
  5. **6 lượt `DFT 10%` (`seed 0..2`, `--no-save-ckpt`)** & **6 lượt `Seq 10%` (`seed 0..2`, `--no-save-ckpt`)**.
  6. **6 lượt `ZS` (`SA/SB × seed 0..2`, `FP32`)** qua `src.phase2.task11 xlmr-zs`.
  7. **Benchmark suy luận chuẩn hóa** cho **cả `ViSoBERT` và `XLM-R-base`** trên cùng GPU, tổng hợp bảng CSV + báo cáo.

### Bảo vệ dung lượng SSD `/content` & Google Drive phẳng
- Mỗi `work/<run_id>/resume.pt` của `XLM-R` (~278M tham số + AdamW FP32) nặng ~**4,4 GB**. Runner tự động xóa `resume.pt` **ngay sau khi từng lượt đạt `complete=True`** (giữ nguyên `provenance.json`) và dùng `--no-save-ckpt` cho toàn bộ 24 lượt `sweep`/`10%`, giữ SSD luôn an toàn.
- Mọi artifact & checkpoint `DFT 100%` được đồng bộ nguyên tử lên thư mục Drive phẳng `/content/drive/MyDrive/Xdomain/Phase2/Task11_XLMR` để phục hồi (`hydrate`) và `SKIP` nếu phiên Colab bị ngắt.

## 1. Tham số, profile `colab-task06-v1` & đường dẫn Drive phẳng

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, platform, re, shutil, signal, subprocess, sys, time

T0_SETUP = time.perf_counter()
SESSION_ID = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())

REPO_URL = "https://github.com/PhuongNgoVan/Xdomain-Transfer-Learning.git"
REPO_REF = "phase2_task11-xlmr-llm"  # đổi sang commit SHA 40 ký tự khi nối lại phiên cũ
PRIVATE_REPO = False

PROFILE_ID = "colab-task06-v1"
PROFILES = {
    "colab-task06-v1": {
        "python": "3.13.18",
        "torch": "2.14.1+cu130",
        "cuda": "13.0",
        "pins": {
            "transformers": "4.57.1",
            "huggingface-hub": "0.36.2",
            "tokenizers": "0.22.2",
            "numpy": "2.1.3",
            "pandas": "3.1.0",
            "scikit-learn": "1.6.1",
            "scipy": "1.16.3",
            "PyYAML": "6.0.2",
            "sentencepiece": "0.2.2",
            "protobuf": "7.36.2",
            "safetensors": "0.8.0",
            "tqdm": "4.67.3",
        },
    },
}
assert PROFILE_ID in PROFILES, f"Unknown PROFILE_ID: {PROFILE_ID}"

MOUNT_DRIVE = True
USE_DRIVE_BACKUP = True
DRIVE_BACKUP_BASE = Path("/content/drive/MyDrive/Xdomain/Phase2/Task11_XLMR")
# Tùy chọn đường dẫn kết quả Task 08 & Task 11 LLM trên Drive để ghép bảng tổng hợp ở Bước 10:
DRIVE_TASK08_RESULTS_DIR = "/content/drive/MyDrive/Xdomain/Phase2/Task08"
DRIVE_TASK11_LLM_DIR = "/content/drive/MyDrive/Xdomain/Phase2/Task11_LLM"

BACKUP_DFT100_CKPTS_TO_DRIVE = True
SAVE_10PCT_MAIN_CKPTS = False
CLEAN_COMPLETED_RESUME_PT = True  # bắt buộc True cho XLM-R (~4.4 GB/run) để không tràn SSD Colab

assert Path("/content").is_dir(), "Notebook này yêu cầu môi trường Google Colab (/content)."
REPO_DIR = Path("/content/xdomain-task11-xlmr")
VENV_DIR = Path(f"/content/phase2-task11-xlmr-{PROFILE_ID}-venv")
SMOKE_DIR = Path("/content/task11_xlmr_smoke")
OUTPUT_BASE = Path("/content/task11_xlmr_outputs")
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
SMOKE_DIR.mkdir(parents=True, exist_ok=True)
print(
    f"Session: {SESSION_ID} | Profile: {PROFILE_ID} | "
    f"Backup DFT100 ckpts: {BACKUP_DFT100_CKPTS_TO_DRIVE} | Clean resume.pt: {CLEAN_COMPLETED_RESUME_PT}"
)

## 2. Kiểm tra GPU CUDA/BF16 ở kernel & mount Google Drive

In [ ]:
import torch

assert torch.cuda.is_available(), "Chưa bật GPU CUDA trên Colab (Runtime -> Change runtime type -> GPU)."
assert torch.cuda.is_bf16_supported(), "GPU hiện tại không hỗ trợ BF16 strict; hãy chọn A100 hoặc L4."

gpu_name = torch.cuda.get_device_name(0)
vram_gib = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2)
print(f"GPU: {gpu_name} | VRAM (GiB): {vram_gib}")
print(
    f"Kernel Python: {platform.python_version()} | "
    f"Preinstalled Torch: {torch.__version__} | CUDA: {torch.version.cuda} | BF16: True"
)

expected_profile = PROFILES[PROFILE_ID]
assert torch.__version__ == expected_profile["torch"], (
    f"Torch trên Colab ({torch.__version__}) khác profile {PROFILE_ID} ({expected_profile['torch']})."
)
assert torch.version.cuda == expected_profile["cuda"], (
    f"CUDA của Torch ({torch.version.cuda}) khác profile {PROFILE_ID} ({expected_profile['cuda']})."
)
subprocess.run(["nvidia-smi"], check=True)

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

## 3. Fetch repo, khóa `COMMIT`, chụp `frozen_before.json` & khôi phục từ Drive phẳng

In [ ]:
import base64

def sha256_path(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def atomic_copy_file(src: Path, dst: Path) -> None:
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + f".{os.getpid()}.tmp")
    shutil.copy2(src, tmp)
    tmp.replace(dst)

def git_command(args, cwd=None) -> str:
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    if PRIVATE_REPO:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        assert token, "Thiếu Colab Secret GITHUB_TOKEN khi PRIVATE_REPO=True."
        auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
        env.update(
            GIT_CONFIG_COUNT="1",
            GIT_CONFIG_KEY_0="http.extraHeader",
            GIT_CONFIG_VALUE_0="Authorization: Basic " + auth,
        )
    result = subprocess.run(
        ["git", *args], cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    if result.returncode != 0:
        raise RuntimeError(
            "Git thất bại: kiểm tra URL, quyền đọc hoặc commit SHA đã được push chưa.\n" + result.stderr
        )
    return result.stdout.strip()

def hydrate_from_flat_drive(
    drive_root: Path,
    results_dir: Path,
    ckpt_out_dir: Path,
    zs_out_dir: Path,
    commit: str,
    profile_id: str,
) -> set[str]:
    drive_env_json = drive_root / "environment" / "environment.json"
    if drive_env_json.is_file():
        d_env = json.loads(drive_env_json.read_text(encoding="utf-8"))
        assert d_env.get("commit") == commit, (
            f"Drive backup {drive_root} thuộc commit khác ({d_env.get('commit')} != {commit})."
        )
        assert d_env.get("profile_id") == profile_id, (
            f"Drive backup {drive_root} thuộc profile khác ({d_env.get('profile_id')} != {profile_id})."
        )

    for sel_name in ("selection_xlmr_dft_lr.json", "selection_xlmr_lr.json"):
        d_sel = drive_root / sel_name
        l_sel = results_dir / sel_name
        if d_sel.is_file() and not l_sel.is_file():
            atomic_copy_file(d_sel, l_sel)

    verified_run_ids: set[str] = set()
    drive_csv = drive_root / "phase2_runs.csv"
    if drive_csv.is_file() and not (results_dir / "phase2_runs.csv").is_file():
        with drive_csv.open("r", encoding="utf-8", newline="") as handle:
            reader = csv.DictReader(handle)
            fieldnames = reader.fieldnames
            rows = list(reader)
        if rows:
            assert {r["git_commit"] for r in rows} <= {commit}, "Backup phase2_runs.csv thuộc commit khác."
            for run_id in sorted({r["run_id"] for r in rows}):
                d_run = drive_root / "runs" / run_id
                d_prov = drive_root / "work" / run_id / "provenance.json"
                d_comp = d_run / "completion.json"
                if not (d_comp.is_file() and d_prov.is_file()):
                    continue
                comp = json.loads(d_comp.read_text(encoding="utf-8"))
                prov = json.loads(d_prov.read_text(encoding="utf-8"))
                if comp.get("complete") is not True or comp.get("fingerprint") != prov.get("fingerprint"):
                    continue
                artifacts = comp.get("artifacts", {})
                ckpt_rel_prefix = comp.get("ckpt_path")
                ckpt_files_expected = list(comp.get("checkpoint_files") or [])
                ckpt_prefix = (str(ckpt_rel_prefix).rstrip("/") + "/") if ckpt_rel_prefix is not None else ""
                copy_plan: list[tuple[Path, Path]] = []
                covered: set[str] = set()
                ok = bool(artifacts) and (ckpt_rel_prefix is None or bool(ckpt_files_expected))
                for rel, digest in artifacts.items():
                    if not rel.startswith("../"):
                        src_f, dst_f = d_run / rel, results_dir / "runs" / run_id / rel
                    elif ckpt_prefix and rel.startswith(ckpt_prefix):
                        sub_rel = rel[len(ckpt_prefix):]
                        covered.add(sub_rel)
                        src_f, dst_f = drive_root / "checkpoints" / run_id / sub_rel, ckpt_out_dir / run_id / sub_rel
                    else:
                        ok = False
                        break
                    if not src_f.is_file() or sha256_path(src_f) != digest:
                        ok = False
                        break
                    copy_plan.append((src_f, dst_f))
                if ok and ckpt_rel_prefix is not None and not set(ckpt_files_expected).issubset(covered):
                    ok = False
                if not ok:
                    continue
                atomic_copy_file(d_prov, results_dir / "work" / run_id / "provenance.json")
                for src_f, dst_f in copy_plan:
                    atomic_copy_file(src_f, dst_f)
                atomic_copy_file(d_comp, results_dir / "runs" / run_id / "completion.json")
                verified_run_ids.add(run_id)
            if verified_run_ids:
                kept = [r for r in rows if r["run_id"] in verified_run_ids]
                tmp_csv = results_dir / "phase2_runs.csv.tmp"
                with tmp_csv.open("w", encoding="utf-8", newline="") as handle:
                    writer = csv.DictWriter(handle, fieldnames=fieldnames, lineterminator="\n")
                    writer.writeheader()
                    writer.writerows(kept)
                tmp_csv.replace(results_dir / "phase2_runs.csv")
                print(f"Hydrated {len(verified_run_ids)} verified Task11 XLM-R runs from Drive.")

    d_zs_csv = drive_root / "xlmr_zs" / "xlmr_zs_runs.csv"
    if d_zs_csv.is_file() and not (zs_out_dir / "xlmr_zs_runs.csv").is_file():
        with d_zs_csv.open("r", encoding="utf-8", newline="") as handle:
            reader = csv.DictReader(handle)
            zs_fields = reader.fieldnames
            zs_rows = list(reader)
        kept_zs_ids: set[str] = set()
        for run_id in sorted({r["run_id"] for r in zs_rows}):
            d_run = drive_root / "xlmr_zs" / "runs" / run_id
            d_prov = drive_root / "xlmr_zs" / "work" / run_id / "provenance.json"
            d_comp = d_run / "completion.json"
            if not (d_comp.is_file() and d_prov.is_file()):
                continue
            comp = json.loads(d_comp.read_text(encoding="utf-8"))
            prov = json.loads(d_prov.read_text(encoding="utf-8"))
            if comp.get("complete") is not True or comp.get("fingerprint") != prov.get("fingerprint"):
                continue
            arts = comp.get("artifacts", {})
            if all((d_run / rel).is_file() and sha256_path(d_run / rel) == dig for rel, dig in arts.items()):
                atomic_copy_file(d_prov, zs_out_dir / "work" / run_id / "provenance.json")
                for rel in arts:
                    atomic_copy_file(d_run / rel, zs_out_dir / "runs" / run_id / rel)
                atomic_copy_file(d_comp, zs_out_dir / "runs" / run_id / "completion.json")
                kept_zs_ids.add(run_id)
        if kept_zs_ids:
            tmp_zs = zs_out_dir / "xlmr_zs_runs.csv.tmp"
            with tmp_zs.open("w", encoding="utf-8", newline="") as handle:
                writer = csv.DictWriter(handle, fieldnames=zs_fields, lineterminator="\n")
                writer.writeheader()
                writer.writerows([r for r in zs_rows if r["run_id"] in kept_zs_ids])
            tmp_zs.replace(zs_out_dir / "xlmr_zs_runs.csv")
            print(f"Hydrated {len(kept_zs_ids)} verified XLM-R ZS runs from Drive.")
    return verified_run_ids

if not REPO_DIR.exists():
    git_command(["clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)])
assert (REPO_DIR / ".git").is_dir(), f"REPO_DIR không phải Git checkout: {REPO_DIR}"
if git_command(["diff", "--name-only"], REPO_DIR) or git_command(["diff", "--cached", "--name-only"], REPO_DIR):
    raise RuntimeError("Checkout có thay đổi cục bộ; hãy dùng REPO_DIR sạch.")

git_command(["fetch", "--depth=1", "origin", REPO_REF], REPO_DIR)
git_command(["checkout", "--detach", "FETCH_HEAD"], REPO_DIR)
COMMIT = git_command(["rev-parse", "HEAD"], REPO_DIR)
assert len(COMMIT) == 40, f"Commit SHA không hợp lệ: {COMMIT}"

OUTPUT_DIR = OUTPUT_BASE / COMMIT[:12] / PROFILE_ID
RESULTS_DIR = OUTPUT_DIR / "results" / "phase2"
CKPT_OUT_DIR = OUTPUT_DIR / "checkpoints" / "phase2"
ZS_OUT_DIR = RESULTS_DIR / "xlmr_zs"
ENV_DIR = OUTPUT_DIR / "environment"
LOG_DIR = OUTPUT_DIR / "logs"
for folder in (RESULTS_DIR, CKPT_OUT_DIR, ZS_OUT_DIR, ENV_DIR, LOG_DIR):
    folder.mkdir(parents=True, exist_ok=True)

DRIVE_SESSION_ROOT = DRIVE_BACKUP_BASE if (MOUNT_DRIVE and USE_DRIVE_BACKUP) else None
if DRIVE_SESSION_ROOT is not None:
    DRIVE_SESSION_ROOT.mkdir(parents=True, exist_ok=True)

required_repo_files = [
    "configs/phase2.yaml",
    "configs/task11_xlmr_smoke.yaml",
    "configs/task11_xlmr_sweep.yaml",
    "src/phase2/cli.py",
    "src/phase2/run.py",
    "src/phase2/models.py",
    "src/phase2/protocol.py",
    "src/phase2/select_lr.py",
    "src/phase2/task11.py",
    "data/clean/neuesc_100.csv",
    "data/clean/vsmec_train.csv",
    "data/clean/vsmec_dev.csv",
    "data/clean/vsmec_test.csv",
    "data/subsets_phase2/MANIFEST.json",
    *[f"data/subsets_phase2/neuesc_f10_s{s}_ids.csv" for s in range(3)],
    *[f"data/subsets_phase2/vsmec_f10_s{s}_ids.csv" for s in range(3)],
]
for rel in required_repo_files:
    assert (REPO_DIR / rel).is_file(), f"Thiếu file bắt buộc tại commit {COMMIT}: {rel}"

frozen_paths = [
    p
    for folder in ("data/clean", "data/splits", "data/subsets_phase2", "src/seq", "src/zs", "src/stats", "src/phase2")
    for p in sorted((REPO_DIR / folder).rglob("*"))
    if p.is_file() and "__pycache__" not in p.parts
]
frozen_paths += [
    REPO_DIR / rel
    for rel in ("configs/base.yaml", "configs/zs.yaml", "configs/seq.yaml", "configs/phase2.yaml", "src/common/constants.py")
]
frozen_snapshot = {p.relative_to(REPO_DIR).as_posix(): sha256_path(p) for p in frozen_paths}
frozen_file = ENV_DIR / "frozen_before.json"
if frozen_file.is_file():
    assert json.loads(frozen_file.read_text(encoding="utf-8")) == frozen_snapshot, "Tài nguyên đóng băng đã thay đổi!"
else:
    frozen_file.write_text(json.dumps(frozen_snapshot, sort_keys=True, indent=2) + "\n", encoding="utf-8")

if DRIVE_SESSION_ROOT is not None:
    hydrate_from_flat_drive(DRIVE_SESSION_ROOT, RESULTS_DIR, CKPT_OUT_DIR, ZS_OUT_DIR, COMMIT, PROFILE_ID)

print("Commit:", COMMIT)
print("Output dir:", OUTPUT_DIR)
print("Drive backup dir:", DRIVE_SESSION_ROOT)

## 4. Kiểm tra `venv`/`ensurepip` & dọn virtualenv lỗi

In [ ]:
probe = subprocess.run([sys.executable, "-c", "import venv, ensurepip"], capture_output=True, text=True)
if probe.returncode != 0:
    py_mm = f"{sys.version_info.major}.{sys.version_info.minor}"
    apt_pkg = f"python{py_mm}-venv"
    print(f"Thiếu venv/ensurepip cho Python {py_mm}; đang cài gói hệ thống {apt_pkg}...")
    apt_upd = subprocess.run(["apt-get", "update", "-qq"], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    apt_ins = subprocess.run(
        ["apt-get", "install", "-y", "-qq", apt_pkg], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    apt_log = (apt_upd.stdout or "") + "\n" + (apt_ins.stdout or "")
    (LOG_DIR / f"apt_venv_{SESSION_ID}.log").write_text(apt_log, encoding="utf-8")
    print(apt_log[-3000:])
    apt_upd.check_returncode()
    apt_ins.check_returncode()
    subprocess.run([sys.executable, "-c", "import venv, ensurepip"], check=True)

sys_py_ver = subprocess.check_output([sys.executable, "-c", "import platform; print(platform.python_version())"], text=True).strip()
print(f"Host sys.executable Python: {sys_py_ver} (profile expects venv Python: {PROFILES[PROFILE_ID]['python']})")

PYTHON = VENV_DIR / "bin" / "python"
PIP_BIN = VENV_DIR / "bin" / "pip"
if VENV_DIR.exists() and not (PYTHON.is_file() and PIP_BIN.is_file()):
    print("Phát hiện virtualenv tạo dở (thiếu python hoặc pip), đang xóa để tạo lại:", VENV_DIR)
    shutil.rmtree(VENV_DIR)
print("Venv support check: PASS")

## 5. Tạo virtualenv, cài 12 package pin & cấu hình `RUN_ENV`

In [ ]:
if not PYTHON.is_file():
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(VENV_DIR)], check=True)
assert PYTHON.is_file() and PIP_BIN.is_file(), f"Tạo virtualenv thất bại: {VENV_DIR}"

pins_list = [f"{name}=={ver}" for name, ver in PROFILES[PROFILE_ID]["pins"].items()]
install = subprocess.run(
    [str(PYTHON), "-m", "pip", "install", *pins_list],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
for install_path in (LOG_DIR / f"install_{SESSION_ID}.log", LOG_DIR / "install.log"):
    install_path.write_text(install.stdout, encoding="utf-8")
print(install.stdout[-4000:])
install.check_returncode()

freeze_out = subprocess.check_output([str(PYTHON), "-m", "pip", "freeze"], text=True)
for freeze_path in (ENV_DIR / f"pip-freeze_{SESSION_ID}.txt", ENV_DIR / "pip-freeze.txt"):
    freeze_path.write_text(freeze_out, encoding="utf-8")

pip_chk = subprocess.run(
    [str(PYTHON), "-m", "pip", "check"], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False
)
pip_chk_text = (pip_chk.stdout or "") + f"\n[pip_check_exit_code]={pip_chk.returncode}\n"
for chk_path in (ENV_DIR / f"pip_check_{SESSION_ID}.txt", ENV_DIR / "pip_check.txt"):
    chk_path.write_text(pip_chk_text, encoding="utf-8")

RUN_ENV = os.environ.copy()
RUN_ENV.update(
    PYTHONUNBUFFERED="1",
    TOKENIZERS_PARALLELISM="false",
    HF_HUB_DISABLE_TELEMETRY="1",
    USE_TF="0",
    USE_TORCH="1",
    TF_CPP_MIN_LOG_LEVEL="3",
)
RUN_ENV.pop("HF_HUB_OFFLINE", None)
RUN_ENV.pop("TRANSFORMERS_OFFLINE", None)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        RUN_ENV["HF_TOKEN"] = hf_token.strip()
        print("Loaded optional HF_TOKEN from Colab Secrets (redacted).")
except Exception:
    pass

def run_python(source: str) -> None:
    result = subprocess.run(
        [str(PYTHON), "-u", "-c", source],
        cwd=REPO_DIR,
        env=RUN_ENV,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    result.check_returncode()

print(f"Installed {len(pins_list)} pinned packages into {VENV_DIR} (pip check exit={pip_chk.returncode}).")

## 6. Preflight môi trường & thử forward/loss BF16 cho `XLM-R-base`

In [ ]:
preflight_env_script = r'''
import importlib.metadata as metadata
import json, platform, subprocess, sys
from pathlib import Path
import numpy, pandas, scipy, sklearn, torch, transformers, yaml
from src.phase2.models import build_phase2_model
from src.phase2.protocol import load_phase2_config
from src.phase2.run import atomic_json

ctx = json.loads(__CTX_JSON__)
profile_spec = ctx["profile_spec"]
env_dir = Path(ctx["env_dir"])

actual_pins = {}
mismatches = []
for pkg, exp_ver in profile_spec["pins"].items():
    try:
        got_ver = metadata.version(pkg)
    except metadata.PackageNotFoundError:
        got_ver = None
    actual_pins[pkg] = got_ver
    if got_ver != exp_ver:
        mismatches.append(f"{pkg}: expected {exp_ver}, got {got_ver}")

py_ver = platform.python_version()
if py_ver != profile_spec["python"]:
    mismatches.append(f"python: expected {profile_spec['python']}, got {py_ver}")
if torch.__version__ != profile_spec["torch"]:
    mismatches.append(f"torch: expected {profile_spec['torch']}, got {torch.__version__}")
if torch.version.cuda != profile_spec["cuda"]:
    mismatches.append(f"cuda: expected {profile_spec['cuda']}, got {torch.version.cuda}")
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    mismatches.append("CUDA BF16 is not available")

try:
    smi_out = subprocess.check_output(["nvidia-smi"], text=True)
except Exception as exc:
    smi_out = f"unavailable: {exc}"

snapshot = {
    "profile_id": ctx["profile_id"],
    "session_id": ctx["session_id"],
    "commit": ctx["commit"],
    "python": py_ver,
    "sys_executable": sys.executable,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "bf16_supported": bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "vram_total_bytes": int(torch.cuda.get_device_properties(0).total_memory) if torch.cuda.is_available() else 0,
    "nvidia_smi": smi_out,
    "pinned_packages_expected": profile_spec["pins"],
    "pinned_packages_actual": actual_pins,
    "preflight_mismatches": mismatches,
    "preflight_status": "FAIL" if mismatches else "ENV_PASS",
}
session_env_file = env_dir / f"environment_{ctx['session_id']}.json"
atomic_json(session_env_file, snapshot)
atomic_json(env_dir / "environment.json", snapshot)
if mismatches:
    raise AssertionError("Profile preflight FAILED: " + "; ".join(mismatches))

cfg = load_phase2_config("configs/phase2.yaml")
model, tok, dft_prov = build_phase2_model("dft", "SA", "XLMR", 0, 10, cfg)
model = model.to("cuda").train()
enc = tok(["kiểm tra tiền huấn luyện một", "kiểm tra tiền huấn luyện hai"], padding=True, truncation=True, max_length=32, return_tensors="pt")
batch = {k: v.to("cuda") for k, v in enc.items()}
batch["labels"] = torch.tensor([0, 2], dtype=torch.long, device="cuda")
with torch.autocast("cuda", dtype=torch.bfloat16):
    out = model(**batch)
assert out.logits.shape == (2, 3) and bool(torch.isfinite(out.loss)), "XLMR DFT BF16 forward/loss preflight failed!"
del out, batch, enc, model, tok
torch.cuda.empty_cache()

snapshot["xlmr_pretrained_preflight"] = {"status": "PASS", "provenance": dft_prov}
snapshot["preflight_status"] = "PASS"
atomic_json(session_env_file, snapshot)
atomic_json(env_dir / "environment.json", snapshot)
print("GPU:", snapshot["gpu"], "| Venv Python:", py_ver)
print("Environment & XLMR BF16 preflight: PASS")
'''

ctx_payload = json.dumps({
    "profile_id": PROFILE_ID,
    "profile_spec": PROFILES[PROFILE_ID],
    "session_id": SESSION_ID,
    "commit": COMMIT,
    "env_dir": str(ENV_DIR),
})
run_python(preflight_env_script.replace("__CTX_JSON__", repr(ctx_payload)))

## 7. Kiểm tra dữ liệu `10%`/`100%` & chạy smoke test tách biệt (`dev-only`)

In [ ]:
data_preflight_script = r'''
import json
from src.phase2.data import load_phase2_prepared_data
from src.phase2.protocol import load_phase2_config

cfg = load_phase2_config("configs/phase2.yaml")
expected = {"SA": {10: (2305, 3305, 6613), 100: (23048, 3305, 6613)},
            "SB": {10: (554, 686, 692), 100: (5542, 686, 692)}}
for direction, by_frac in expected.items():
    for frac, (exp_tr, exp_dv, exp_te) in by_frac.items():
        for seed in range(3):
            p = load_phase2_prepared_data(direction, frac, seed, "XLMR", cfg, run_mode="main")
            assert len(p["train_df"]) == exp_tr and len(p["dev_df"]) == exp_dv and len(p["test_df"]) == exp_te
            assert p["provenance"]["test_integrity_checked"] is True
    print(f"Data integrity ({direction}, 10%/100%, s0..2): PASS")
'''
run_python(data_preflight_script)

smoke_res = SMOKE_DIR / "results"
smoke_ckpt = SMOKE_DIR / "ckpt"
smoke_cmd = [
    str(PYTHON), "-u", "-m", "src.phase2.cli",
    "--config", "configs/task11_xlmr_smoke.yaml",
    "--max-examples", "50",
    "--results-dir", str(smoke_res),
    "--ckpt-out-dir", str(smoke_ckpt),
]
smoke_proc = subprocess.run(smoke_cmd, cwd=REPO_DIR, env=RUN_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(LOG_DIR / f"smoke_{SESSION_ID}.log").write_text(smoke_proc.stdout, encoding="utf-8")
print(smoke_proc.stdout[-2500:])
smoke_proc.check_returncode()
assert not list((smoke_res / "runs").glob("*/predictions_test.csv")), "Smoke dev-only không được tạo predictions_test.csv!"
for rpt in (smoke_res / "work").glob("*/resume.pt"):
    rpt.unlink()

if DRIVE_SESSION_ROOT is not None:
    for folder_name in ("environment", "logs"):
        for item in sorted((OUTPUT_DIR / folder_name).rglob("*")):
            if item.is_file() and item.suffix != ".tmp":
                atomic_copy_file(item, DRIVE_SESSION_ROOT / folder_name / item.relative_to(OUTPUT_DIR / folder_name))
    print("Synced preflight env & smoke logs to Drive:", DRIVE_SESSION_ROOT)

## 8. Thực thi tuần tự `DFT sweep → DFT 100% → Seq sweep → DFT/Seq 10% → ZS`

In [ ]:
TRAIN_LOG_PATH = LOG_DIR / f"train_xlmr_{SESSION_ID}.log"
COMMANDS_RECORD = []

def stop_process_group(proc: subprocess.Popen) -> None:
    if proc.poll() is None:
        try:
            os.killpg(proc.pid, signal.SIGINT)
            proc.wait(timeout=60)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGTERM)
            proc.wait(timeout=30)
        except ProcessLookupError:
            pass

def sync_completed_run_to_drive(run_id: str, save_ckpt_flag: bool) -> None:
    if DRIVE_SESSION_ROOT is None:
        return
    d_res = DRIVE_SESSION_ROOT
    atomic_copy_file(RESULTS_DIR / "work" / run_id / "provenance.json", d_res / "work" / run_id / "provenance.json")
    if save_ckpt_flag and BACKUP_DFT100_CKPTS_TO_DRIVE:
        local_ckpt_dir = CKPT_OUT_DIR / run_id
        drive_ckpt_dir = d_res / "checkpoints" / run_id
        for item in sorted(local_ckpt_dir.rglob("*")):
            if item.is_file() and item.suffix != ".tmp":
                dst = drive_ckpt_dir / item.relative_to(local_ckpt_dir)
                if dst.is_file() and dst.stat().st_size == item.stat().st_size and sha256_path(dst) == sha256_path(item):
                    continue
                atomic_copy_file(item, dst)
    local_run_dir = RESULTS_DIR / "runs" / run_id
    for item in sorted(local_run_dir.rglob("*")):
        if item.is_file() and item.name != "completion.json" and item.suffix != ".tmp":
            atomic_copy_file(item, d_res / "runs" / run_id / item.relative_to(local_run_dir))
    atomic_copy_file(local_run_dir / "completion.json", d_res / "runs" / run_id / "completion.json")
    atomic_copy_file(RESULTS_DIR / "phase2_runs.csv", d_res / "phase2_runs.csv")

def run_cli_job(stage_tag: str, idx: int, total: int, cmd: list[str], save_ckpt_flag: bool) -> str:
    COMMANDS_RECORD.append({"stage": stage_tag, "index": idx, "argv": cmd})
    (LOG_DIR / f"commands_{SESSION_ID}.json").write_text(
        json.dumps(COMMANDS_RECORD, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    with TRAIN_LOG_PATH.open("a", encoding="utf-8") as log_handle:
        start_offset = log_handle.tell()
        log_handle.write(f"\n=== [{stage_tag} {idx}/{total}] start {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())} ===\n")
        log_handle.flush()
        t_job = time.perf_counter()
        proc = subprocess.Popen(
            cmd, cwd=REPO_DIR, env=RUN_ENV, stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True
        )
        last_hb = time.monotonic()
        try:
            while proc.poll() is None:
                time.sleep(3)
                if time.monotonic() - last_hb >= 30:
                    last_hb = time.monotonic()
                    tail = TRAIN_LOG_PATH.read_text(encoding="utf-8", errors="replace")[-400:]
                    last_ln = [ln for ln in tail.splitlines() if ln.strip()][-1:] or ["..."]
                    print(f"  [{stage_tag} {idx}/{total} running {round(time.perf_counter() - t_job)}s] {last_ln[0]}")
        except BaseException:
            stop_process_group(proc)
            if DRIVE_SESSION_ROOT is not None and TRAIN_LOG_PATH.is_file():
                atomic_copy_file(TRAIN_LOG_PATH, DRIVE_SESSION_ROOT / "logs" / TRAIN_LOG_PATH.name)
            raise

    with TRAIN_LOG_PATH.open("r", encoding="utf-8", errors="replace") as rh:
        rh.seek(start_offset)
        segment = rh.read()
    if proc.returncode != 0:
        if DRIVE_SESSION_ROOT is not None and TRAIN_LOG_PATH.is_file():
            atomic_copy_file(TRAIN_LOG_PATH, DRIVE_SESSION_ROOT / "logs" / TRAIN_LOG_PATH.name)
        print(segment[-5000:])
        raise RuntimeError(f"[{stage_tag} {idx}/{total}] thất bại (exit={proc.returncode}).")

    matches = re.findall(r"^([A-Za-z0-9._\-]+): (complete|SKIP)\s*$", segment, flags=re.MULTILINE)
    assert matches, f"Không tìm thấy '<run_id>: complete|SKIP' ở [{stage_tag} {idx}/{total}].\n{segment[-1500:]}"
    run_id, status = matches[-1]
    assert (RESULTS_DIR / "work" / run_id / "provenance.json").is_file()
    if CLEAN_COMPLETED_RESUME_PT:
        rpt = RESULTS_DIR / "work" / run_id / "resume.pt"
        if rpt.is_file():
            rpt.unlink()
    sync_completed_run_to_drive(run_id, save_ckpt_flag)
    m_obj = json.loads((RESULTS_DIR / "runs" / run_id / "metrics.json").read_text(encoding="utf-8"))
    c_obj = json.loads((RESULTS_DIR / "runs" / run_id / "cost.json").read_text(encoding="utf-8"))
    print(f"[{stage_tag} {idx}/{total}] {run_id} | {status} | dev={m_obj['dev_selection_score']:.4f} | steps={c_obj['actual_update_steps']} | s={c_obj['train_seconds']:.1f}")
    return run_id

# 8A. 6 DFT sweep runs (10%, s0, dev-only) -> lock selection_xlmr_dft_lr.json
dft_sweeps = [(d, lr) for d in ("SA", "SB") for lr in (2e-5, 3e-5, 5e-5)]
for i, (d, lr) in enumerate(dft_sweeps, 1):
    run_cli_job("8A-DFT-SWEEP", i, 6, [
        str(PYTHON), "-u", "-m", "src.phase2.cli",
        "--config", "configs/task11_xlmr_sweep.yaml",
        "--direction", d, "--method", "dft", "--lr", f"{lr:g}",
        "--results-dir", str(RESULTS_DIR), "--ckpt-out-dir", str(CKPT_OUT_DIR), "--no-save-ckpt",
    ], False)

subprocess.run([
    str(PYTHON), "-u", "-m", "src.phase2.select_lr",
    "--results-dir", str(RESULTS_DIR), "--backbone", "XLMR", "--methods", "dft",
], cwd=REPO_DIR, env=RUN_ENV, check=True)
SEL_DFT_PATH = RESULTS_DIR / "selection_xlmr_dft_lr.json"
sel_dft = json.loads(SEL_DFT_PATH.read_text(encoding="utf-8"))
if DRIVE_SESSION_ROOT is not None:
    atomic_copy_file(SEL_DFT_PATH, DRIVE_SESSION_ROOT / SEL_DFT_PATH.name)
    atomic_copy_file(RESULTS_DIR / "xlmr_dft_lr_selection.csv", DRIVE_SESSION_ROOT / "xlmr_dft_lr_selection.csv")
print("Locked DFT LRs:", {k: v["lr"] for k, v in sel_dft["selected"].items()})

# 8B. 6 DFT 100% runs (s0..2, save_ckpt=True, --selection-path)
dft100_jobs = [(d, s, float(sel_dft["selected"][f"XLMR/{d}/dft"]["lr"])) for d in ("SA", "SB") for s in range(3)]
for i, (d, s, lr) in enumerate(dft100_jobs, 1):
    run_cli_job("8B-DFT-100", i, 6, [
        str(PYTHON), "-u", "-m", "src.phase2.cli",
        "--config", "configs/phase2.yaml",
        "--backbone", "XLMR", "--direction", d, "--method", "dft", "--fraction", "100",
        "--seed", str(s), "--lr", f"{lr:g}", "--run-mode", "main",
        "--selection-path", str(SEL_DFT_PATH),
        "--results-dir", str(RESULTS_DIR), "--ckpt-out-dir", str(CKPT_OUT_DIR),
    ], True)

# 8C. 6 Seq sweep runs (10%, s0, dev-only) -> lock selection_xlmr_lr.json
seq_sweeps = [(d, lr) for d in ("SA", "SB") for lr in (1e-5, 2e-5, 3e-5)]
for i, (d, lr) in enumerate(seq_sweeps, 1):
    run_cli_job("8C-SEQ-SWEEP", i, 6, [
        str(PYTHON), "-u", "-m", "src.phase2.cli",
        "--config", "configs/task11_xlmr_sweep.yaml",
        "--direction", d, "--method", "seq", "--lr", f"{lr:g}",
        "--source-ckpt-root", str(CKPT_OUT_DIR),
        "--results-dir", str(RESULTS_DIR), "--ckpt-out-dir", str(CKPT_OUT_DIR), "--no-save-ckpt",
    ], False)

subprocess.run([
    str(PYTHON), "-u", "-m", "src.phase2.select_lr",
    "--results-dir", str(RESULTS_DIR), "--backbone", "XLMR", "--methods", "dft,seq",
], cwd=REPO_DIR, env=RUN_ENV, check=True)
SEL_FULL_PATH = RESULTS_DIR / "selection_xlmr_lr.json"
sel_full = json.loads(SEL_FULL_PATH.read_text(encoding="utf-8"))
if DRIVE_SESSION_ROOT is not None:
    atomic_copy_file(SEL_FULL_PATH, DRIVE_SESSION_ROOT / SEL_FULL_PATH.name)
    atomic_copy_file(RESULTS_DIR / "xlmr_lr_selection.csv", DRIVE_SESSION_ROOT / "xlmr_lr_selection.csv")
print("Locked full XLM-R LRs:", {k: v["lr"] for k, v in sel_full["selected"].items()})

# 8D. 6 DFT 10% + 8E. 6 Seq 10% (s0..2, --no-save-ckpt)
main10_jobs = [
    (d, m, s, float(sel_full["selected"][f"XLMR/{d}/{m}"]["lr"]))
    for d in ("SA", "SB") for m in ("dft", "seq") for s in range(3)
]
for i, (d, m, s, lr) in enumerate(main10_jobs, 1):
    cmd = [
        str(PYTHON), "-u", "-m", "src.phase2.cli",
        "--config", "configs/phase2.yaml",
        "--backbone", "XLMR", "--direction", d, "--method", m, "--fraction", "10",
        "--seed", str(s), "--lr", f"{lr:g}", "--run-mode", "main",
        "--selection-path", str(SEL_FULL_PATH),
        "--results-dir", str(RESULTS_DIR), "--ckpt-out-dir", str(CKPT_OUT_DIR),
    ]
    if m == "seq":
        cmd += ["--source-ckpt-root", str(CKPT_OUT_DIR)]
    if not SAVE_10PCT_MAIN_CKPTS:
        cmd.append("--no-save-ckpt")
    run_cli_job("8DE-MAIN-10", i, 12, cmd, SAVE_10PCT_MAIN_CKPTS)

# 8F. 6 ZS inference runs (FP32) via src.phase2.task11 xlmr-zs
zs_res = subprocess.run([
    str(PYTHON), "-u", "-m", "src.phase2.task11", "xlmr-zs",
    "--config", "configs/phase2.yaml",
    "--results-dir", str(RESULTS_DIR),
    "--ckpt-dir", str(CKPT_OUT_DIR),
    "--out-dir", str(ZS_OUT_DIR),
    "--batch-size", "32",
], cwd=REPO_DIR, env=RUN_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(zs_res.stdout)
zs_res.check_returncode()

if DRIVE_SESSION_ROOT is not None:
    for item in sorted(ZS_OUT_DIR.rglob("*")):
        if item.is_file() and item.suffix != ".tmp":
            atomic_copy_file(item, DRIVE_SESSION_ROOT / "xlmr_zs" / item.relative_to(ZS_OUT_DIR))
print("Completed all 30 train + 6 ZS XLM-R runs.")

## 9. Kiểm tra `SKIP` & nghiệm thu độc lập toàn bộ `30` lượt train + `6` `ZS`

In [ ]:
sel_full = json.loads((RESULTS_DIR / "selection_xlmr_lr.json").read_text(encoding="utf-8"))["selected"]
for d in ("SA", "SB"):
    for frac, m in ((100, "dft"), (10, "dft"), (10, "seq")):
        lr = float(sel_full[f"XLMR/{d}/{m}"]["lr"])
        cmd = [
            str(PYTHON), "-u", "-m", "src.phase2.cli",
            "--config", "configs/phase2.yaml",
            "--backbone", "XLMR", "--direction", d, "--method", m, "--fraction", str(frac),
            "--seed", "0", "--lr", f"{lr:g}", "--run-mode", "main",
            "--selection-path", str(RESULTS_DIR / "selection_xlmr_lr.json"),
            "--results-dir", str(RESULTS_DIR), "--ckpt-out-dir", str(CKPT_OUT_DIR),
        ]
        if m == "seq":
            cmd += ["--source-ckpt-root", str(CKPT_OUT_DIR)]
        if frac == 10 and not SAVE_10PCT_MAIN_CKPTS:
            cmd.append("--no-save-ckpt")
        res = subprocess.run(cmd, cwd=REPO_DIR, env=RUN_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        print(res.stdout.strip())
        res.check_returncode()
        assert ": SKIP" in res.stdout

zs_skip = subprocess.run([
    str(PYTHON), "-u", "-m", "src.phase2.task11", "xlmr-zs",
    "--config", "configs/phase2.yaml", "--results-dir", str(RESULTS_DIR),
    "--ckpt-dir", str(CKPT_OUT_DIR), "--out-dir", str(ZS_OUT_DIR), "--batch-size", "32",
], cwd=REPO_DIR, env=RUN_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(zs_skip.stdout.strip())
zs_skip.check_returncode()
assert zs_skip.stdout.count(": SKIP") == 6

verify_xlmr_script = r'''
import json
from pathlib import Path
import numpy as np
import pandas as pd
from src.common.hashing import sha256_file
from src.common.schema import ZS_PRED_COLUMNS
from src.phase2.protocol import canonical_digest

ctx = json.loads(__CTX_JSON__)
results_dir = Path(ctx["results_dir"])
zs_dir = Path(ctx["zs_dir"])
env_dir = Path(ctx["env_dir"])
commit = ctx["commit"]

manifests = sorted((results_dir / "runs").glob("*/completion.json"))
assert len(manifests) == 30, f"Cần 30 train completion.json (12 sweep + 18 main), có {len(manifests)}"
code_digests = set()
for mp in manifests:
    rdir = mp.parent
    comp = json.loads(mp.read_text(encoding="utf-8"))
    assert comp.get("complete") is True
    for rel, dig in comp["artifacts"].items():
        target = (rdir / rel).resolve()
        assert target.is_file() and sha256_file(target) == dig
    prov = json.loads((results_dir / "work" / rdir.name / "provenance.json").read_text(encoding="utf-8"))
    code_digests.add(canonical_digest(prov["source_code_sha256"]))
    meta = json.loads((rdir / "metadata.json").read_text(encoding="utf-8"))
    if meta["run_mode"] == "main":
        pred = pd.read_csv(rdir / "predictions_test.csv")
        assert tuple(pred.columns) == ZS_PRED_COLUMNS and pred["id"].is_unique
        p3 = pred[["p3_neg", "p3_neu", "p3_pos"]].to_numpy(float)
        assert np.isfinite(p3).all() and (p3 >= 0).all() and np.allclose(p3.sum(axis=1), 1.0, atol=1e-6)
        if meta["direction"] == "SB":
            assert int(pred["in_strict"].astype(bool).sum()) == 526
assert len(code_digests) == 1, "source_code_sha256 không đồng nhất!"

zs_manifests = sorted((zs_dir / "runs").glob("*/completion.json"))
assert len(zs_manifests) == 6, f"Cần 6 ZS completion.json, có {len(zs_manifests)}"
for mp in zs_manifests:
    comp = json.loads(mp.read_text(encoding="utf-8"))
    assert comp.get("complete") is True
    for rel, dig in comp["artifacts"].items():
        assert sha256_file(mp.parent / rel) == dig

rows = pd.read_csv(results_dir / "phase2_runs.csv", keep_default_na=False)
assert rows["run_id"].nunique() == 30 and len(rows) == 108 and rows["git_commit"].eq(commit).all()
zs_rows = pd.read_csv(zs_dir / "xlmr_zs_runs.csv", keep_default_na=False)
assert zs_rows["run_id"].nunique() == 6 and len(zs_rows) == 12

frozen = json.loads((env_dir / "frozen_before.json").read_text(encoding="utf-8"))
assert all(sha256_file(Path(rel)) == dig for rel, dig in frozen.items()), "Tài nguyên đóng băng bị thay đổi!"
print("Independent verification of all 30 XLM-R train runs, 6 ZS runs, SKIP & frozen files: PASS")
'''
run_python(verify_xlmr_script.replace("__CTX_JSON__", repr(json.dumps({
    "results_dir": str(RESULTS_DIR), "zs_dir": str(ZS_OUT_DIR), "env_dir": str(ENV_DIR), "commit": COMMIT
}))))

## 10. Benchmark suy luận `ViSoBERT` vs `XLM-R`, xuất bảng tổng hợp & đồng bộ Drive

In [ ]:
COST_ENC_PATH = RESULTS_DIR / "cost_encoder_inference.json"
bench_res = subprocess.run([
    str(PYTHON), "-u", "-m", "src.phase2.task11", "benchmark-encoders",
    "--config", "configs/phase2.yaml",
    "--precision", "fp32",
    "--n-b1", "100",
    "--large-batch-size", "32",
    "--out-path", str(COST_ENC_PATH),
], cwd=REPO_DIR, env=RUN_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(bench_res.stdout)
bench_res.check_returncode()

sum_cmd = [
    str(PYTHON), "-u", "-m", "src.phase2.task11", "summarize",
    "--config", "configs/phase2.yaml",
    "--xlmr-results-dir", str(RESULTS_DIR),
    "--xlmr-zs-dir", str(ZS_OUT_DIR),
    "--encoder-cost-path", str(COST_ENC_PATH),
    "--out-dir", str(RESULTS_DIR / "tables"),
    "--report-path", str(OUTPUT_DIR / "reports" / "phase2" / "task11_xlmr_llm_10pct.md"),
]
if MOUNT_DRIVE and DRIVE_TASK08_RESULTS_DIR.strip() and Path(DRIVE_TASK08_RESULTS_DIR.strip()).is_dir():
    sum_cmd += ["--visobert-results-dir", DRIVE_TASK08_RESULTS_DIR.strip()]
if MOUNT_DRIVE and DRIVE_TASK11_LLM_DIR.strip() and Path(DRIVE_TASK11_LLM_DIR.strip()).is_dir():
    sum_cmd += ["--llm-results-dir", DRIVE_TASK11_LLM_DIR.strip(), "--llm-cost-dir", DRIVE_TASK11_LLM_DIR.strip()]

sum_res = subprocess.run(sum_cmd, cwd=REPO_DIR, env=RUN_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(sum_res.stdout)
sum_res.check_returncode()
print((OUTPUT_DIR / "reports" / "phase2" / "task11_xlmr_llm_10pct.md").read_text(encoding="utf-8"))

if DRIVE_SESSION_ROOT is not None:
    must_copy = [
        (COST_ENC_PATH, DRIVE_SESSION_ROOT / "cost_encoder_inference.json"),
        (OUTPUT_DIR / "reports" / "phase2" / "task11_xlmr_llm_10pct.md", DRIVE_SESSION_ROOT / "task11_xlmr_llm_10pct.md"),
        (RESULTS_DIR / "phase2_runs.csv", DRIVE_SESSION_ROOT / "phase2_runs.csv"),
    ]
    for tbl in sorted((RESULTS_DIR / "tables").glob("*.csv")):
        must_copy.append((tbl, DRIVE_SESSION_ROOT / "tables" / tbl.name))
    for folder_name in ("environment", "logs"):
        for item in sorted((OUTPUT_DIR / folder_name).rglob("*")):
            if item.is_file() and item.suffix != ".tmp":
                must_copy.append((item, DRIVE_SESSION_ROOT / folder_name / item.relative_to(OUTPUT_DIR / folder_name)))
    for src_f, dst_f in must_copy:
        atomic_copy_file(src_f, dst_f)
        assert dst_f.is_file() and sha256_path(dst_f) == sha256_path(src_f), f"SHA-256 mismatch: {dst_f}"
    print("Synced & SHA-256 verified all Task 11 XLM-R deliverables on flat Drive:", DRIVE_SESSION_ROOT)